# Mallory, Hardcastle et al. (2021): the freely moving claims, on the paper's figshare deposits

**Paper.** Mallory CS, Hardcastle K, Campbell MG, Attinger A, Low IIC, Raymond JL, Giocomo LM (2021).
Mouse entorhinal cortex encodes a diverse repertoire of self-motion signals. *Nature Communications*
12, 671. [doi:10.1038/s41467-021-20936-8](https://doi.org/10.1038/s41467-021-20936-8). The claims file
(`claims/mallory-2021.yaml`) was written from the version of record with its Methods, legends and
Supplementary Information.

**Data.** Dandiset 000053 holds no head pitch or roll, so a reader marked the freely moving claims
`data: elsewhere` (question of 2026-11-10), on the paper's own deposits: figshare Dataset 1
([doi:10.6084/m9.figshare.13425392.v1](https://doi.org/10.6084/m9.figshare.13425392.v1),
Freely_moving_data_with_inertial_sensor.mat, Cohort 1, 179 cells with pitch and roll) and Dataset 2
([doi:10.6084/m9.figshare.13425017.v1](https://doi.org/10.6084/m9.figshare.13425017.v1),
Freely_moving_data_without_inertial_sensor.mat, Cohort 2). Both are version 1, the only version.
Dataset 1 is one compressed MATLAB 5 variable, so it is streamed whole into memory (1.7 GB) and
reduced as it loads; Dataset 2 (6.4 GB, MATLAB 7.3) is read in place, only its azimuthal head
velocity, for Figure 1b. Nothing is written to disk. Loading is in `mec_free.py`, which also holds the open field LN model
(not run here, see the table).

| Claims | Status |
|---|---|
| fig1b-head-posture-distributions | tested here (Datasets 1 and 2) |
| the claims that need the LN model on Cohort 1 (fig1c, fig1d, fig2 pitch and roll, supp4-head, supp5-head) | blocked: the LN model under the four readings did not finish within the 60-minute limit of a notebook cell (see the attempt record) |
| the claims that need the LN model on Cohort 2 (fig1c-azimuthal-velocity-cells and the fig2c and fig3 claims on it, supp9-head) | blocked: too large for the time limit |
| supp3-head-low-false-positives | blocked: six more runs of the Cohort 1 LN model |
| head-fixed claims | `reproduce_mallory_2021.ipynb` |

In [ ]:
import math
import os
import time
import resource
import warnings
from collections import Counter

import numpy as np
import pandas as pd
from scipy import stats

import mec_ln as L
import mec_free as F

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
STARTED = time.time()
SUBSET = int(os.environ.get("MALLORY_SUBSET", "0"))   # 0: every session (develop on a few)


def gb():
    return round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2)

## Data check

In [ ]:
t0 = time.time()
raw = F.fetch()
print(f"Dataset 1: {len(raw) / 1e9:.3f} GB streamed in {time.time() - t0:.0f} s")
SESS = F.load_dataset1(raw)
del raw
if SUBSET:
    SESS = dict(list(SESS.items())[:SUBSET])
print(f"read in {time.time() - t0:.0f} s; peak memory {gb()} GB")
rows = []
for sid, s in SESS.items():
    b = s["beh"]
    dt = np.diff(s["t"])
    rows.append({"session": sid, "mouse": s["animal"], "arena_cm": s["arena"], "cells": len(s["cells"]),
                 "minutes": round(s["t"][-1] / 60, 1), "dt_min": round(dt.min(), 4), "dt_max": round(dt.max(), 4),
                 "same_behaviour": s["same_behaviour"],
                 "x_range": (round(np.nanmin(b["x"]), 1), round(np.nanmax(b["x"]), 1)),
                 "y_range": (round(np.nanmin(b["y"]), 1), round(np.nanmax(b["y"]), 1)),
                 "nan%": round(100 * max(np.isnan(b[k]).mean() for k in ("x", "y", "Bs", "Ha", "Hp", "Hr")), 2)})
S1 = pd.DataFrame(rows)
print(S1.to_string(index=False))
print("\nCells:", int(S1.cells.sum()), "in", len(S1), "sessions of", S1.mouse.nunique(), "mice;",
      S1.groupby("mouse").cells.sum().to_dict())
print("Every cell of a session carries the same behaviour:", bool(S1.same_behaviour.all()))
print("Arena sides (cm):", dict(Counter(S1.arena_cm)))
allb = {k: np.concatenate([s["beh"][k] for s in SESS.values()]) for k in ("x", "y", "Bs", "Ha", "Had", "Hp", "Hr", "Hpd", "Hrd")}
print("\nRanges over all sessions (1st, 50th, 99th percentile; min, max):")
units = {"x": "cm", "y": "cm", "Bs": "cm/s", "Ha": "rad", "Had": "deg/s", "Hp": "deg", "Hr": "deg", "Hpd": "deg/s", "Hrd": "deg/s"}
for k, v in allb.items():
    print(f"  {k:4s} [{units[k]}]: {np.round(np.nanpercentile(v, [1, 50, 99]), 2)}; {np.nanmin(v):.2f} to {np.nanmax(v):.2f}")
n_sp = np.concatenate([s["n_spikes"] for s in SESS.values()])
n_in = np.concatenate([s["Y"].sum(1) for s in SESS.values()])
print("Spikes outside the behaviour samples:", int((n_sp - n_in).sum()), "of", int(n_sp.sum()))

The deposit's description gives positions in cm, speed in cm/s, head direction in degrees, azimuthal
head velocity in deg/s and pitch and roll in radians. The values agree except for head direction,
which is in radians, not degrees: its 1st to 99th percentiles are 0.05 to 6.23 (a few samples lie
outside 0 to 2 pi, -1.67 to 8.42). Pitch and roll, converted to degrees, have 1st to 99th
percentiles of -41.8 to 68.1 and -69.0 to 75.7 deg. Behaviour is sampled every 20 ms (the Methods'
50 Hz) in every session. The arenas are 50 and 70 cm square, two of the sizes the Methods list. No
spike falls outside the behaviour samples.

In [ ]:
# How the deposit's azimuthal head velocity relates to its head direction, and so how to take the
# pitch and roll velocities the deposit lacks.
s = next(iter(SESS.values()))
ha = np.degrees(s["beh"]["Ha"])
fwd = np.diff(ha) / F.DT
print("Azimuthal head velocity equal to the forward difference of head direction (deg / 20 ms), "
      "where the angle does not wrap:")
nowrap = np.abs(np.diff(ha)) < 180
print("  fraction of samples:", round(float(np.mean(np.isclose(s["beh"]["Had"][:-1][nowrap], fwd[nowrap], atol=1e-6))), 4))
wrap = ~nowrap
print("  where it wraps, |velocity| > 9000 deg/s in", round(float(np.mean(np.abs(s["beh"]["Had"][:-1][wrap]) > 9000)), 3),
      "of", int(wrap.sum()), "samples (the wrap is not undone; these lie outside the inner 95%)")
# Sessions that share their behaviour
keys = list(SESS)
dups = []
for i, a in enumerate(keys):
    for b in keys[i + 1:]:
        A, B = SESS[a]["beh"], SESS[b]["beh"]
        if len(A["x"]) == len(B["x"]) and np.array_equal(A["x"], B["x"], equal_nan=True):
            dups.append((a, b, {k: bool(np.array_equal(A[k], B[k], equal_nan=True)) for k in ("y", "Bs", "Ha", "Hp", "Hr")}))
print("\nSessions with identical body position:", dups)

The deposit's azimuthal head velocity is the forward difference of its head direction, not unwrapped,
so the velocities of pitch and roll are taken the same way (`mec_free`). Two sessions of different
mice (Fuschia 052001 and Daisy 052001) share their body position, speed and head direction exactly,
with different pitch and roll; this is reported as a finding, and both are kept, as the deposit keeps
them.

In [ ]:
# Clock check: spikes, LED tracking and the IMU share a clock if the tuning to body position and to
# pitch is strongest with no shift. Fraction of the variance of each cell's spike count (smoothed
# over 0.5 s) explained by 10 x 10 position bins, or 20 pitch bins, with the behaviour shifted.
LAGS = np.arange(-100, 101, 10)   # 20 ms bins: -2 to +2 s


def eta2(lab, y, nlab):
    n = np.bincount(lab, minlength=nlab)
    s = np.bincount(lab, weights=y, minlength=nlab)
    ok = n > 0
    tot = ((y - y.mean()) ** 2).sum()
    return float(((s[ok] ** 2 / n[ok]).sum() - y.sum() ** 2 / len(y)) / tot) if tot > 0 else np.nan


best = {"B": [], "Hp": []}
for sid, s in SESS.items():
    b = s["beh"]
    ok = np.isfinite(b["x"]) & np.isfinite(b["y"]) & np.isfinite(b["Hp"])
    ix = np.clip(((b["x"] - np.nanmin(b["x"])) / np.ptp(b["x"][ok]) * 10).astype(int), 0, 9)
    iy = np.clip(((b["y"] - np.nanmin(b["y"])) / np.ptp(b["y"][ok]) * 10).astype(int), 0, 9)
    lp = np.percentile(b["Hp"][ok], [1, 99])
    ip = np.clip(((b["Hp"] - lp[0]) / (lp[1] - lp[0]) * 20).astype(int), 0, 19)
    for yc in s["Y"]:
        ys = np.convolve(yc.astype(float), np.ones(25) / 25, "same")
        for var, lab, nl in (("B", ix * 10 + iy, 100), ("Hp", ip, 20)):
            e = []
            for lag in LAGS:
                sl = slice(max(0, lag), len(ys) + min(0, lag))
                sb = slice(max(0, -lag), len(ys) - max(0, lag))
                m = ok[sb]
                e.append(eta2(lab[sb][m], ys[sl][m], nl))
            best[var].append((sid, LAGS[int(np.nanargmax(e))], max(e)))
for var, rows_ in best.items():
    lag = np.array([r[1] for r in rows_]) * F.DT
    top = np.array([r[2] for r in rows_])
    strong = top > np.percentile(top, 75)
    print(f"{var}: lag of best tuning over {len(lag)} cells, median {np.median(lag):+.2f} s; "
          f"cells with the strongest quarter of tuning: median {np.median(lag[strong]):+.2f} s, "
          f"{np.mean(np.abs(lag[strong]) <= 0.2):.2f} within 0.2 s")
    by = pd.DataFrame(rows_, columns=["session", "lag", "eta2"])
    by["mouse"] = by.session.map(lambda x: SESS[x]["animal"])
    print("   median lag by mouse (s):", (by.groupby("mouse").lag.median() * F.DT).round(2).to_dict())

The check does not establish a shared clock. Over the 179 cells the shift that gives the strongest
tuning has a median of -0.20 s for position and -0.40 s for pitch, and among the quarter of cells
with the strongest tuning only 0.18 (position) and 0.24 (pitch) peak within 0.2 s of no shift; the
median shift differs by mouse (-0.4 to +1.4 s for position). With 20 ms bins and a 0.5 s smoothing
of the spike counts this measure is coarse, and most cells are weakly tuned, so it neither shows nor
rules out an offset. The one claim tested here, fig1b, uses the behaviour alone, so it does not
depend on the clock.

In [ ]:
# Dataset 2: cells, sessions and mice, read in place.
t0 = time.time()
IDX2 = pd.DataFrame(F.dataset2_index())
print(f"Dataset 2 index read in {time.time() - t0:.0f} s")
print("Cells:", len(IDX2), "; sessions:", IDX2.session.nunique(), "; mice:", IDX2.animal.nunique())
print("Arena sides (cm), by cell:", IDX2.arena.value_counts().to_dict())
print("Mice in both datasets:", sorted(set(IDX2.animal) & set(S1.mouse)))
first = IDX2.groupby("session").row.first()
if SUBSET:
    first = first.iloc[:SUBSET * 2]
t0 = time.time()
HAV2 = F.dataset2_series(first.tolist(), "azimuthal_head_velocity")
print(f"azimuthal head velocity of {len(HAV2)} sessions read in {time.time() - t0:.0f} s; "
      f"1st and 99th percentile over all: {np.round(np.nanpercentile(np.concatenate(list(HAV2.values())), [1, 99]), 1)} deg/s")

### fig1b-head-posture-distributions

> Across Cohort 1 mice the median head pitch is 11.5 deg (inner 95% from -23.0 to 55.6 deg) and the
> median roll -0.8 deg (-50.9 to 43.6 deg); azimuthal head velocity has a median near 0 deg/s and an
> inner 95% from -389.9 to 385.1 deg/s across 25 mice of both cohorts.

Percentiles per session, averaged over each mouse's sessions, then the mean over mice, against the
legend's sem across mice.

In [ ]:
rows = []
for sid, s in SESS.items():
    for v in ("Hp", "Hr", "Had"):
        p = np.nanpercentile(s["beh"][v], [2.5, 50, 97.5])
        rows.append({"mouse": s["animal"], "session": sid, "var": v, "p2.5": p[0], "p50": p[1], "p97.5": p[2], "cohort": 1})
an2 = IDX2.groupby("session").animal.first()
for sid, row in first.items():
    p = np.nanpercentile(HAV2[row], [2.5, 50, 97.5])
    rows.append({"mouse": an2[sid], "session": sid, "var": "Had", "p2.5": p[0], "p50": p[1], "p97.5": p[2], "cohort": 2})
pct = pd.DataFrame(rows)
per_mouse = pct.groupby(["var", "mouse"])[["p2.5", "p50", "p97.5"]].mean()
EXPECTED_1B = [("Hp", "p50", 11.5, 4.1), ("Hp", "p2.5", -23.0, 4.5), ("Hp", "p97.5", 55.6, 4.1),
               ("Hr", "p50", -0.8, 7.6), ("Hr", "p2.5", -50.9, 3.6), ("Hr", "p97.5", 43.6, 4.0),
               ("Had", "p2.5", -389.9, 34.7), ("Had", "p97.5", 385.1, 34.7)]
out = []
for v, q, val, tol in EXPECTED_1B:
    x = per_mouse.loc[v][q]
    out.append({"var": v, "percentile": q, "mean over mice": round(x.mean(), 2), "sem": round(x.sem(), 2),
                "mice": len(x), "paper": val, "tol": tol, "within": abs(x.mean() - val) <= tol})
t1b = pd.DataFrame(out)
print(t1b.to_string(index=False))
print("Within tolerance:", int(t1b.within.sum()), "of 8")
print("Azimuthal head velocity median, mean over mice:", round(per_mouse.loc["Had"]["p50"].mean(), 2), "deg/s")
print("\nPer mouse (deg, deg/s):")
print(per_mouse.round(1).to_string())
print("Sessions per mouse:", pct[pct["var"] == "Had"].groupby("mouse").session.size().to_dict())

**Result: reproduced.** All eight values are within the legend's sem: pitch median 11.45, 2.5th
percentile -23.03 and 97.5th percentile 55.57 deg over 6 mice; roll -0.76, -50.92 and 43.63 deg;
azimuthal head velocity 2.5th and 97.5th percentiles -389.82 and 385.01 deg/s over the 25 mice of
both cohorts, with a median of -0.1 deg/s.

## Summary

| Claim | Outcome |
|---|---|
| fig1b-head-posture-distributions | reproduced |
| Cohort 1 LN claims (fig1c-head-direction-cells, fig1c-pitch-roll-cells, fig1c-pitch-roll-velocity-rare, fig1d-conjunctive-head, fig2-pitch-roll-shapes, fig2a, fig2b, fig2-pitch-roll-stability, supp4-head, supp5-head) | blocked, too large: the LN model under the four readings did not finish within the 60-minute cell limit |
| Cohort 2 LN claims (fig1c-azimuthal-velocity-cells, the five fig2c claims, fig3ab, fig3c, fig3e, supp9-head) | blocked, too large |
| supp3-head-low-false-positives | blocked, too large |

**How the analysis differs from the paper's.** Percentiles are taken over every behaviour sample of a
session, as the legend describes; nothing else is assumed.

**What was found about the data.** The deposit's head direction is in radians, not the degrees its
description gives. Two sessions of different mice (Fuschia 052001 and Daisy 052001) have identical
body position, speed and head direction with different pitch and roll. The check of a shared clock
between spikes and behaviour was inconclusive.

In [ ]:
print(f"Notebook time: {(time.time() - STARTED) / 60:.1f} min; peak memory {gb()} GB")